# 🧬 GIADA Task 17a — sostituzione causale Ca_HVA
Questa è la prima verifica causale nel teacher completo. Tre bracci appaiati: nativo, copia-formula, LUT-513 congelata. Nessun voltaggio futuro in ingresso; `gate_c_authorized` rimane falso anche se il microcanary passa. Il protocollo preregistrato è in `experiments/teacher_roadmap_task17_causal_replacement_preregistration_v1.md`.


In [ ]:
from pathlib import Path
import base64,json,os,shutil,subprocess,sys,zipfile
from IPython.display import Javascript,display
WORK=Path('/kaggle/working/giada_roadmap_task17');REPO=WORK/'giada';TEACHER=WORK/'neuron_as_deep_net'
assert not WORK.exists(),f'Workspace già esistente: {WORK}. Avvia una sessione pulita.'
WORK.mkdir(parents=True)
subprocess.run(['git','clone','https://github.com/Zagred47/giada.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'fetch','origin','codex/surrogate-validity-audit'],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach','FETCH_HEAD'],check=True)
subprocess.run(['git','clone','https://github.com/SelfishGene/neuron_as_deep_net.git',str(TEACHER)],check=True)
subprocess.run(['git','-C',str(TEACHER),'checkout','--detach','074c4666300a8ad246601dab179a97a6942f0f29'],check=True)
REVISION=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
assert (REPO/'src/giada_teacher/roadmap_causal_cahva_replacement.py').is_file(),'Revisione GIADA senza Task 17a.'
print({'giada_revision':REVISION,'teacher_revision':'074c4666300a8ad246601dab179a97a6942f0f29'})


## 🧰 Preflight NEURON prima della simulazione
Richiede NEURON 8.2.7; i meccanismi canonici restano invariati. I due cloni Task 17 vengono compilati separatamente dal runner.


In [ ]:
try:
 import neuron
except ModuleNotFoundError:
 subprocess.run([sys.executable,'-m','pip','install','--quiet','neuron==8.2.7'],check=True)
 import neuron
assert neuron.__version__.split('+',1)[0]=='8.2.7',f'NEURON {neuron.__version__} non è 8.2.7.'
nrnivmodl=shutil.which('nrnivmodl') or str(Path(sys.executable).parent/'nrnivmodl')
assert Path(nrnivmodl).is_file() and shutil.which('gcc'),'Compilatore NEURON/gcc assente.'
simulation_dir=TEACHER/'L5PC_NEURON_simulation'
if not list(simulation_dir.rglob('libnrnmech.so')):subprocess.run([nrnivmodl,'mods'],cwd=simulation_dir,check=True)
assert list(simulation_dir.rglob('libnrnmech.so')),'Meccanismi canonici non compilati.'
print({'neuron':neuron.__version__,'canonical_mechanisms':'ready'})


## 📁 Input e contratto selezionato
Aggiungi l'artefatto `giada_roadmap_task15c_embedded_interface_bridge.zip` come Dataset Kaggle. Non serve l'HDF5 targeted da 6 GiB: qui creiamo nuovi seed e traiettorie dal teacher. Se il mount non viene trovato, imposta `GIADA_TASK15C_ARTIFACT` al file ZIP o alla cartella estratta.


In [ ]:
sys.path.insert(0,str(REPO))
from src.giada_teacher.roadmap_embedded_frozen_confirmation import verified_15c
INPUT=Path('/kaggle/input');override=os.environ.get('GIADA_TASK15C_ARTIFACT')
candidates=[Path(override).expanduser()] if override else []
if INPUT.is_dir():
 candidates += list(INPUT.rglob('*.zip'))
 candidates += [p.parent for p in INPUT.rglob('final_report.json') if 'task15c' in str(p).lower() or 'interface_bridge' in str(p).lower()]
TASK15C=None
for candidate in dict.fromkeys(candidates):
 if not candidate.exists():continue
 try:verified_15c(candidate);TASK15C=candidate.resolve();break
 except (ValueError,RuntimeError,FileNotFoundError,OSError,zipfile.BadZipFile):continue
assert TASK15C is not None,'Artefatto Task 15c esatto non trovato negli Input Kaggle.'
print({'task15c':str(TASK15C),'selection':'lut_fine_path'})


## 🔬 Microcanary accoppiato
Il runner salva un `failure_report.json` anche se un preflight fallisce. I risultati restano diagnostici finché Gate C non viene confermato in uno studio più ampio.


In [ ]:
OUTPUT=Path('/kaggle/working/artifacts/giada_roadmap_task17_causal_replacement')
assert not OUTPUT.exists(),f'Output già presente: {OUTPUT}. Avvia una sessione pulita.'
command=[sys.executable,str(REPO/'scripts/run_roadmap_task17.py'),'--repo',str(REPO),'--teacher',str(TEACHER),'--task15c',str(TASK15C),'--output',str(OUTPUT)]
completed=subprocess.run(command,check=False)
result_path=OUTPUT/('final_report.json' if (OUTPUT/'final_report.json').is_file() else 'failure_report.json')
assert result_path.is_file(),'Nessun report: controlla il log del sottoprocesso.'
report=json.loads(result_path.read_text())
display({key:report.get(key) for key in ('valid','decision','gate_c_authorized','formula_control_valid','lut_absolute_valid','paired_effect_valid','identifiable_effect_count','episode_count','error_type','error')})
if completed.returncode:print('Preflight o simulazione non superati; scarica comunque il report diagnostico.')


## 📦 Download ZIP
Il file contiene report, metriche e tracciati ridotti a 1 ms; le tracce complete a 0,025 ms restano negli Output Kaggle. È scaricabile anche dopo un NO-GO.


In [ ]:
EXPORT=WORK/'download_report';EXPORT.mkdir()
for name in ('final_report.json','failure_report.json','paired_metrics.json','paired_metrics.partial.jsonl','paired_effects.json','paired_traces_1ms.json','frozen_table_probe.json'):
 source=OUTPUT/name
 if source.is_file():shutil.copy2(source,EXPORT/name)
archive=Path(shutil.make_archive('/kaggle/working/giada_roadmap_task17_causal_replacement','zip',EXPORT.parent,EXPORT.name))
payload=base64.b64encode(archive.read_bytes()).decode('ascii')
display(Javascript(f"""const b=atob('{payload}');const a=new Uint8Array(b.length);for(let i=0;i<b.length;i++)a[i]=b.charCodeAt(i);const u=URL.createObjectURL(new Blob([a],{{type:'application/zip'}}));const l=document.createElement('a');l.href=u;l.download='{archive.name}';document.body.appendChild(l);l.click();l.remove();setTimeout(()=>URL.revokeObjectURL(u),1000);"""))
print({'archive':archive.name,'size_mib':round(archive.stat().st_size/2**20,2)})
